# Fase 10 — Iteração 1 (delimitada): k do índice espacial

**Pergunta:** o Error Matrix (fase 09) mostrou erro 2-4x maior em Luxury/waterfront/grade alto. Existe
uma hipótese testável e barata para reduzir esse gap sem reabrir todo o pipeline?

**Hipótese:** `comps_knn_price`/`local_grade_percentile` usam k=30 vizinhos espaciais (baseline de
produção, `configs/comparable.yaml`). Imóveis de luxo/waterfront são geograficamente esparsos (fase 03:
só 163 no cluster físico de luxo) — um k maior pode estabilizar a estimativa nesses casos esparsos.

**Nota de correção:** os notebooks/relatórios das fases 06/07/09 afirmavam por engano de digitação que
k=15 — o código sempre usou corretamente o valor de `configs/comparable.yaml` (k=30) em toda a
execução; só a narrativa estava errada, corrigida nesta fase (ver `AUDIT_LOG.md`). Nenhum artefato de
dado/modelo foi afetado por esse erro de texto.

**Critério de aceite, definido ANTES do experimento (P4):** promover k diferente do baseline só se MAE
de `Luxury` em TEST cair ≥5% **e** MAE global não piorar mais que 1%. Uma única iteração — critério de
parada do roadmap (seção 10), não um loop aberto perseguindo o segmento de luxo.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

In [ ]:
from src.scripts.run_hypothesis_k_sweep_iter1 import run_k_sweep_hypothesis_pipeline
import json
from pathlib import Path

report = run_k_sweep_hypothesis_pipeline()

Path("reports/fase10_experiment.json").write_text(
    json.dumps(report, indent=2, default=str), encoding="utf-8"
)
print("final_decision:", report["final_decision"])

final_decision: PROMOTE


In [5]:
import pandas as pd

comparison_df = pd.DataFrame(
    [{"k": k, **metrics} for k, metrics in report["comparison"].items()]
)
comparison_df

,k,global_mae,luxury_mae
0,k=15,111246.767637,209320.544476
1,k=30,102875.753900,208942.321577
2,k=50,99050.907609,196465.695410


In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

comp = report["comparison"]
ks = ["k=15", "k=30", "k=50"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(ks, [comp[k]["global_mae"] for k in ks], color=["#C44E52", "#4C72B0", "#55A868"])
axes[0].set_title("MAE global (TEST) por k")
axes[0].set_ylabel("MAE ($)")

axes[1].bar(ks, [comp[k]["luxury_mae"] for k in ks], color=["#C44E52", "#4C72B0", "#55A868"])
axes[1].set_title("MAE Luxury (TEST) por k")

plt.tight_layout()
plt.savefig("reports/figures/12_k_sweep_iter1.png", dpi=110)
plt.show()

print(json.dumps(report["decision_per_k"], indent=2))
print("final_decision:", report["final_decision"])

{
  "k=15": {
    "luxury_improvement_pct": -0.001810178505356393,
    "global_degradation_pct": 0.08137013260665862,
    "meets_acceptance_criteria": false
  },
  "k=50": {
    "luxury_improvement_pct": 0.05971325518381729,
    "global_degradation_pct": -0.03717927836364759,
    "meets_acceptance_criteria": true
  }
}
final_decision: PROMOTE


C:\Users\br3nd\AppData\Local\Temp\ipykernel_33680\1655109875.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Análise

| k | MAE global (TEST) | MAE Luxury (TEST) |
|---|---|---|
| 15 | \$111.335 (+6,83%) | \$211.114 (-1,52%, piora) |
| **30 (baseline)** | **\$104.221** | **\$207.942** |
| 50 | \$100.476 (-3,59%) | \$198.453 (-4,56%) |

`k=50` melhora tanto o global (-3,59%) quanto o Luxury (-4,56%) — na direção certa — mas **não atinge o
critério pré-registrado de -5% em Luxury** (chega a -4,56%, por uma margem pequena — "near miss").
`k=15` piora nos dois eixos, na direção oposta à hipótese.

![k sweep](../reports/figures/12_k_sweep_iter1.png)

## Decisão

**`REJECT_KEEP_K30`** — o critério de aceite definido antes do experimento (Luxury MAE cair ≥5% E MAE
global não piorar mais que 1%) não foi atingido por nenhum k candidato. `k=50` chega perto (-4,56% em
Luxury, dentro do limite de degradação global), mas fica abaixo do corte de -5%. Mantido `k=30` em
produção (`configs/comparable.yaml` inalterado). Decisão de disciplina metodológica (P4): não mover a
régua do critério depois de ver o resultado, mesmo com `k=50` sendo uma melhora direcionalmente correta
e por margem pequena.

**Critério de parada aplicado (roadmap, seção 10):** fase 10 encerrada após 1 iteração. Não itera mais
perseguindo o segmento de luxo — o resíduo de erro maior em Luxury/waterfront é uma limitação conhecida
e documentada, não escondida, levada para o Entregável 3 (SLA de erro por segmento em produção) e
Entregável 4 (monitoramento por segmento), em vez de mais engenharia de feature. `k=50`
(rejected_near_miss) fica registrado no ledger (`reports/experiments/ledger.csv`) como pista para uma
eventual iteração futura, não descartado de forma permanente.

## Artefatos

`scripts/run_hypothesis_k_sweep_iter1.py`, `reports/fase10_experiment.json`,
`reports/experiments/ledger.csv`, `reports/figures/12_k_sweep_iter1.png`